# YOLO Training — Config-Driven
Set `ARGS_YAML_PATH` below and run all cells. All training hyperparameters are read from the YAML file.

In [ ]:
%pip install ultralytics -q

from google.colab import drive, runtime
drive.mount('/content/drive')

from ultralytics import YOLO
from pathlib import Path
from PIL import Image
import shutil
import yaml

TRAIN_DIR_RAW = "/content/drive/MyDrive/CMPE401/YOLO/VisDrone2019-DET-train"
VAL_DIR_RAW   = "/content/drive/MyDrive/CMPE401/YOLO/VisDrone2019-DET-val"
TEST_DIR_RAW  = "/content/drive/MyDrive/CMPE401/YOLO/VisDrone2019-DET-test-dev"

ARGS_YAML_PATH = "/content/drive/MyDrive/CMPE401/YOLO/runs/visdrone_yolo11s_baseline7/args.yaml"

DST_DIR = "/content/drive/MyDrive/CMPE401/YOLO/data"

# Data Formatting

Format the data from the VisDrone dataset into one that the YOLO model can use.

In [ ]:
# ── Class mapping ──────────────────────────────────────────────────────────────
VISDRONE_CLASSES = {
    1: 0,   # pedestrian
    2: 1,   # people
    3: 2,   # bicycle
    4: 3,   # car
    5: 4,   # van
    6: 5,   # truck
    7: 6,   # tricycle
    8: 7,   # awning-tricycle
    9: 8,   # bus
    10: 9,  # motor
    # cat 0  = ignored region  → skipped
    # cat 11 = others          → skipped
}

# ── Converter ──────────────────────────────────────────────────────────────────
def convert_split(src_dir, split_name):
    src_dir    = Path(src_dir)
    images_dir = src_dir / "images"
    annots_dir = src_dir / "annotations"

    out_images = Path(DST_DIR) / "images" / split_name
    out_labels = Path(DST_DIR) / "labels" / split_name
    out_images.mkdir(parents=True, exist_ok=True)
    out_labels.mkdir(parents=True, exist_ok=True)

    all_images = sorted(images_dir.glob("*.jpg"))
    total      = len(all_images)

    print(f"\n{'='*60}")
    print(f"  Converting split : {split_name.upper()}")
    print(f"  Source           : {src_dir}")
    print(f"  Images found     : {total}")

    # ── Cache check: skip images already converted ─────────────────────────────
    pending = []
    already_done = 0
    for img_path in all_images:
        out_img   = out_images / img_path.name
        out_label = out_labels / (img_path.stem + ".txt")
        if out_img.exists() and out_label.exists():
            already_done += 1
        else:
            pending.append(img_path)

    if already_done > 0:
        print(f"  Cache hit        : {already_done} / {total} already converted — skipping")
    if not pending:
        print(f"  ✅ Split '{split_name}' fully cached — nothing to do!")
        print(f"{'='*60}")
        return
    print(f"  To process       : {len(pending)} images")
    print(f"{'='*60}")

    # ── Process only pending images ────────────────────────────────────────────
    skipped_imgs  = 0
    total_boxes   = 0
    skipped_boxes = 0

    for idx, img_path in enumerate(pending, 1):
        annot_path = annots_dir / (img_path.stem + ".txt")

        # Progress every 500 images
        if idx % 500 == 0 or idx == len(pending):
            print(f"  [{idx:>5}/{len(pending)}] Processing {img_path.name} ...")

        if not annot_path.exists():
            print(f"  [WARN] No annotation for {img_path.name} — skipping")
            skipped_imgs += 1
            continue

        img          = Image.open(img_path)
        img_w, img_h = img.size

        yolo_lines = []
        with open(annot_path) as f:
            for line in f:
                parts = line.strip().split(",")
                if len(parts) < 6:
                    skipped_boxes += 1
                    continue

                x, y, w, h = int(parts[0]), int(parts[1]), int(parts[2]), int(parts[3])
                score       = int(parts[4])
                cat         = int(parts[5])

                # Skip ignored regions, 'others', and score=0 (ignored annotations)
                if cat not in VISDRONE_CLASSES or score == 0:
                    skipped_boxes += 1
                    continue

                yolo_cat = VISDRONE_CLASSES[cat]
                cx = max(0.0, min(1.0, (x + w / 2) / img_w))
                cy = max(0.0, min(1.0, (y + h / 2) / img_h))
                nw = max(0.0, min(1.0, w / img_w))
                nh = max(0.0, min(1.0, h / img_h))

                if nw > 0 and nh > 0:
                    yolo_lines.append(f"{yolo_cat} {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}")
                    total_boxes += 1
                else:
                    skipped_boxes += 1

        shutil.copy(img_path, out_images / img_path.name)
        with open(out_labels / (img_path.stem + ".txt"), "w") as f:
            f.write("\n".join(yolo_lines))

    print(f"\n  ✅ Done — {split_name.upper()}")
    print(f"     Images converted : {len(pending) - skipped_imgs} / {len(pending)} pending")
    print(f"     Bounding boxes   : {total_boxes} kept  |  {skipped_boxes} skipped (ignored/invalid)")
    print(f"     Output images    : {out_images}")
    print(f"     Output labels    : {out_labels}")


convert_split(TRAIN_DIR_RAW, "train")
convert_split(VAL_DIR_RAW,   "val")
convert_split(TEST_DIR_RAW,  "test")

print(f"\n{'='*60}")
print("  All splits converted successfully!")
print(f"{'='*60}\n")

In [ ]:
yaml_content = f"""\
path: {DST_DIR}
train: images/train
val:   images/val
test:  images/test

nc: 10
names:
  0: pedestrian
  1: people
  2: bicycle
  3: car
  4: van
  5: truck
  6: tricycle
  7: awning-tricycle
  8: bus
  9: motor
"""

yaml_path = Path(DST_DIR) / "visdrone.yaml"
yaml_path.parent.mkdir(parents=True, exist_ok=True)
with open(yaml_path, "w") as f:
    f.write(yaml_content)
print(f"  YAML saved → {yaml_path}\n")

# Training the Experiment Model

In [ ]:
# ── Load args ──────────────────────────────────────────────────────────────────
args_path = Path(ARGS_YAML_PATH)
assert args_path.exists(), f"args.yaml not found at: {args_path}"

with open(args_path) as f:
    cfg = yaml.safe_load(f)

print(f"Loaded config from: {args_path}")
print(f"  model  : {cfg.get('model')}")
print(f"  data   : {cfg.get('data')}")
print(f"  name   : {cfg.get('name')}")
print(f"  epochs : {cfg.get('epochs')}")
print(f"  imgsz  : {cfg.get('imgsz')}")
print(f"  batch  : {cfg.get('batch')}")

# ── Keys used by YOLO() constructor or set internally by Ultralytics ───────────
CONSTRUCTOR_KEYS = {"model", "task"}
EXCLUDED_KEYS    = {"save_dir", "mode"}

model = YOLO(cfg["model"])

train_kwargs = {
    k: v
    for k, v in cfg.items()
    if k not in CONSTRUCTOR_KEYS and k not in EXCLUDED_KEYS and v is not None
}

results = model.train(**train_kwargs)

# Validation and Statistics

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import pandas as pd
from pathlib import Path

# Derive run_dir from the args.yaml so it stays in sync with whichever experiment you trained
run_dir = Path(cfg["save_dir"])

# ── 1. Print final metrics ──────────────────────────────────────────
metrics = model.val()
print("=" * 40)
print("         BASELINE METRICS")
print("=" * 40)
print(f"mAP@50:       {metrics.box.map50:.4f}")
print(f"mAP@50-95:    {metrics.box.map:.4f}")
print(f"Precision:    {metrics.box.mp:.4f}")
print(f"Recall:       {metrics.box.mr:.4f}")
print("=" * 40)

# ── 2. Plot loss curves from results.csv ───────────────────────────
csv_path = run_dir / 'results.csv'
df = pd.read_csv(csv_path)
df.columns = df.columns.str.strip()  # remove whitespace from column names

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Baseline YOLOv11s — Loss Curves', fontsize=14)

# Training loss
axes[0].plot(df['epoch'], df['train/box_loss'], label='Box Loss')
axes[0].plot(df['epoch'], df['train/cls_loss'], label='Class Loss')
axes[0].plot(df['epoch'], df['train/dfl_loss'], label='DFL Loss')
axes[0].set_title('Training Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True)

# Validation loss
axes[1].plot(df['epoch'], df['val/box_loss'], label='Box Loss')
axes[1].plot(df['epoch'], df['val/cls_loss'], label='Class Loss')
axes[1].plot(df['epoch'], df['val/dfl_loss'], label='DFL Loss')
axes[1].set_title('Validation Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.savefig(run_dir / 'loss_curves.png', dpi=150)
plt.show()

# ── 3. Plot mAP over epochs ────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(df['epoch'], df['metrics/mAP50(B)'], label='mAP@50')
ax.plot(df['epoch'], df['metrics/mAP50-95(B)'], label='mAP@50-95')
ax.set_title('Baseline YOLOv11s — mAP over Epochs')
ax.set_xlabel('Epoch')
ax.set_ylabel('mAP')
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.savefig(run_dir / 'map_curves.png', dpi=150)
plt.show()

# ── 4. Show confusion matrix and other plots saved by Ultralytics ──
for img_name in ['confusion_matrix.png', 'PR_curve.png', 'F1_curve.png']:
    img_path = run_dir / img_name
    if img_path.exists():
        fig, ax = plt.subplots(figsize=(10, 8))
        ax.imshow(mpimg.imread(img_path))
        ax.axis('off')
        ax.set_title(img_name.replace('.png', '').replace('_', ' '))
        plt.tight_layout()
        plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# Reuse run_dir derived from args.yaml above
df = pd.read_csv(run_dir / "results.csv")
df.columns = df.columns.str.strip()

epochs = df["epoch"]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle("YOLOv11s Baseline — Training & Validation Curves", fontsize=14, fontweight="bold")

# ── Loss curves ────────────────────────────────────────────────────────────────
axes[0, 0].plot(epochs, df["train/box_loss"], label="Train", color="blue")
axes[0, 0].plot(epochs, df["val/box_loss"],   label="Val",   color="orange")
axes[0, 0].set_title("Box Loss")
axes[0, 0].set_xlabel("Epoch")
axes[0, 0].legend()
axes[0, 0].grid(True)

axes[0, 1].plot(epochs, df["train/cls_loss"], label="Train", color="blue")
axes[0, 1].plot(epochs, df["val/cls_loss"],   label="Val",   color="orange")
axes[0, 1].set_title("Class Loss")
axes[0, 1].set_xlabel("Epoch")
axes[0, 1].legend()
axes[0, 1].grid(True)

axes[0, 2].plot(epochs, df["train/dfl_loss"], label="Train", color="blue")
axes[0, 2].plot(epochs, df["val/dfl_loss"],   label="Val",   color="orange")
axes[0, 2].set_title("DFL Loss")
axes[0, 2].set_xlabel("Epoch")
axes[0, 2].legend()
axes[0, 2].grid(True)

# ── Metric curves ──────────────────────────────────────────────────────────────
axes[1, 0].plot(epochs, df["metrics/mAP50(B)"],    label="mAP@50",    color="green")
axes[1, 0].plot(epochs, df["metrics/mAP50-95(B)"], label="mAP@50-95", color="purple")
axes[1, 0].set_title("mAP over Epochs")
axes[1, 0].set_xlabel("Epoch")
axes[1, 0].legend()
axes[1, 0].grid(True)

axes[1, 1].plot(epochs, df["metrics/precision(B)"], label="Precision", color="red")
axes[1, 1].plot(epochs, df["metrics/recall(B)"],    label="Recall",    color="brown")
axes[1, 1].set_title("Precision & Recall")
axes[1, 1].set_xlabel("Epoch")
axes[1, 1].legend()
axes[1, 1].grid(True)

# ── Train vs Val box loss gap (overfitting indicator) ──────────────────────────
gap = df["val/box_loss"] - df["train/box_loss"]
axes[1, 2].plot(epochs, gap, label="Val - Train Box Loss", color="darkred")
axes[1, 2].axhline(0, color="black", linestyle="--", linewidth=0.8)
axes[1, 2].set_title("Generalization Gap (Val − Train Loss)")
axes[1, 2].set_xlabel("Epoch")
axes[1, 2].legend()
axes[1, 2].grid(True)

plt.tight_layout()
plot_path = run_dir / "loss_analysis.png"
plt.savefig(plot_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved → {plot_path}")